# Step 2 — Waveform Preprocessing Pipeline

**Goal:** Take `benchmark_manifest.csv` (4,476 traces, 8 datasets) and produce
`benchmark_waveforms.hdf5` — a self-contained archive of preprocessed,
model-ready 3-component waveforms at 100 Hz with P/S sample positions.

**Two loading paths — one per data source:**

```
benchmark_manifest.csv
        │
        ├─── Non-MLAAPDE traces (3,126) ──────────────────────────────────────┐
        │     │                                                                │
        │     ▼                                                                │
        │  2.2  Download metadata CSVs  ←── ~2-5 MB each, seconds not hours   │
        │     │                                                                │
        │     ▼                                                                │
        │  2.3  Build FDSN request list  ←── N.S.L.C + time from metadata     │
        │     │                                                                │
        │     ▼                                                                │
        │  2.4  Bulk FDSN fetch  ←── IRIS / NCEDC / SCEDC / INGV              │
        │     │  saves MiniSEED files to waveform_cache/                       │
        │     ▼                                                                │
        └─── 2.6  MiniSEED → preprocess → HDF5 ──────────────────────────────┤
                                                                               │
        ├─── MLAAPDE traces (1,350) ──────────────────────────────────────────┤
        │     │  waveforms already on disk as waveforms_YYYYMM.hdf5           │
        │     ▼  native 40 Hz → resample to 100 Hz                            │
        └─── 2.6b  SeisBench HDF5 → preprocess → HDF5 ───────────────────────┤
                                                                               ▼
                                                            benchmark_waveforms.hdf5
```

**MLAAPDE-specific notes:**
- Waveforms already downloaded locally (sharded by month: `waveforms_YYYYMM.hdf5`)
- Native sampling rate is **40 Hz** — resampled to 100 Hz here
- Mostly P-only (1,349 traces) — window anchored on P; the 1 S-only trace is anchored on S
- FDSN is never called for MLAAPDE traces


## 2.1  Imports & Configuration

Key configuration decisions:
- `TARGET_SR = 100` Hz — PhaseNet's fixed input rate; MLAAPDE (40 Hz) and any other non-100 Hz datasets are resampled here
- `WINDOW_SAMPLES = 3000` — 30 s at 100 Hz, matching PhaseNet's input window
- `P_JITTER_MIN/MAX` — P arrival is placed randomly between 3 s and 27 s into the window so the model cannot learn pick position from window placement
- MLAAPDE is excluded from `DS_FDSN_CLIENT` — its waveforms are already downloaded locally


In [7]:
import numpy as np
import pandas as pd
import h5py, requests, shutil, warnings
from pathlib import Path
from tqdm.notebook import tqdm
import scipy.signal
from obspy import UTCDateTime, Stream
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import FDSNNoDataException

# ── Paths ──────────────────────────────────────────────────────────────────
MANIFEST_PATH  = Path("benchmark_manifest.csv")
METADATA_DIR   = Path("seisbench_metadata")   # metadata CSVs only (MB, not GB)
WAVEFORM_CACHE = Path("waveform_cache")       # per-trace MiniSEED files
OUTPUT_HDF5    = Path("benchmark_waveforms.hdf5")
SEISBENCH_CACHE = Path.home() / ".seisbench" / "datasets"
METADATA_DIR.mkdir(exist_ok=True)
WAVEFORM_CACHE.mkdir(exist_ok=True)

assert MANIFEST_PATH.exists(), "benchmark_manifest.csv not found — run Step 1 first"

# ── Preprocessing parameters ───────────────────────────────────────────────
TARGET_SR      = 100        # Hz — PhaseNet standard
WINDOW_SAMPLES = 3000       # 30 s at 100 Hz
P_JITTER_MIN   = 300        # P at least 3 s into window
P_JITTER_MAX   = 2700       # P at most 27 s into window
FETCH_DURATION = 120        # seconds of waveform to request per FDSN trace
FDSN_BATCH     = 50         # traces per bulk FDSN request
RANDOM_SEED    = 42
rng = np.random.default_rng(RANDOM_SEED)

# ── MLAAPDE-specific constants ─────────────────────────────────────────────
# MLAAPDE waveforms are stored locally at native 40 Hz in monthly shards.
# They are loaded via SeisBench directly — FDSN is never called for MLAAPDE.
MLAAPDE_NATIVE_SR = 40.0
MLAAPDE_DIR       = SEISBENCH_CACHE / "mlaapde"

# ── FDSN data centre per dataset (MLAAPDE excluded — loaded locally) ───────
DS_FDSN_CLIENT = {
    "stead":          "NCEDC",   # Northern California Earthquake Data Center
    "scedc":          "SCEDC",   # Southern California Seismic Network
    "instancecounts": "INGV",    # Istituto Nazionale di Geofisica e Vulcanologia
    "pnw":            "IRIS",    # Pacific Northwest Seismic Network (via IRIS)
    "txed":           "IRIS",    # Texas Seismic Network (via IRIS)
    # mlaapde: NOT listed — waveforms already downloaded locally
}

# ── SeisBench remote metadata URL template ─────────────────────────────────
SEISBENCH_META_URL = (
    "https://dcache-demo.desy.de:2443/Userfiles/seisbench/datasets/{name}/metadata.csv"
)

# ── Load manifest ──────────────────────────────────────────────────────────
manifest = pd.read_csv(MANIFEST_PATH)

# Keep traces with at least one pick (P or S).
# Non-MLAAPDE traces always have P; MLAAPDE may be P-only, S-only, or both.
has_pick = manifest["p_arrival_sample"].notna() | manifest["s_arrival_sample"].notna()
manifest = manifest[has_pick].reset_index(drop=True)

# Ensure has_p_pick column exists (back-compat with older manifests)
if "has_p_pick" not in manifest.columns:
    manifest["has_p_pick"] = manifest["p_arrival_sample"].notna()

print(f"Manifest: {len(manifest):,} traces across {manifest['dataset'].nunique()} datasets")
print(manifest.groupby("dataset")["trace_name"].count().rename("n").to_string())
print(f"\nPick availability:")
print(f"  has P : {manifest['has_p_pick'].sum():,}")
print(f"  has S : {manifest['has_s_pick'].sum():,}")
print(f"  both  : {(manifest['has_p_pick'] & manifest['has_s_pick']).sum():,}")


Manifest: 4,476 traces across 8 datasets
dataset
instancecounts     845
iquique            320
mlaapde           1350
obst2024           400
pnw                378
scedc              326
stead              672
txed               185

Pick availability:
  has P : 4,475
  has S : 3,127
  both  : 3,126


In [8]:
import pandas as pd
from pathlib import Path

CACHE = Path.home() / ".seisbench" / "datasets"
meta  = pd.read_csv(CACHE / "neic" / "metadata.csv", nrows=5)

print("Columns:")
print(list(meta.columns))
print()
print("First row:")
print(meta.iloc[0].to_string())

Columns:
['trace_name', 'trace_category', 'trace_p_arrival_sample', 'trace_p_status', 'source_magnitude', 'source_id', 'path_ep_distance_km', 'path_back_azimuth_deg', 'split', 'trace_name_original', 'trace_s_arrival_sample', 'trace_s_status']

First row:
trace_name                bucket0$0,:3,:2400
trace_category                    earthquake
trace_p_arrival_sample                1200.0
trace_p_status                        manual
source_magnitude                        3.49
source_id                           1000EUZR
path_ep_distance_km                27.798732
path_back_azimuth_deg                   82.3
split                                  train
trace_name_original             1000EUZR_st0
trace_s_arrival_sample                   NaN
trace_s_status                           NaN


## 2.2  Download Metadata CSVs Only

Downloads only the small `metadata.csv` for each non-MLAAPDE dataset — a few MB each,
taking seconds. **The waveforms.hdf5 files (100s of GB) are never downloaded.**

The metadata CSV is needed to recover the FDSN parameters (network, station, channel,
start time) for each trace so we can request just the waveforms we need.

**MLAAPDE is handled differently:** Its metadata is sharded across 16 monthly files
(`metadata_YYYYMM.csv`) already on disk. This cell loads and concatenates all of them
into a single indexed DataFrame — no network download required.


In [9]:
dataset_meta = {}   # ds_name → DataFrame (indexed by trace_name)

for ds_name in manifest["dataset"].unique():

    # ── MLAAPDE: load from local monthly shards ──────────────────────────
    if ds_name == "mlaapde":
        monthly = sorted([f for f in MLAAPDE_DIR.glob("metadata_*.csv")
                          if not f.name.endswith(".partial")])
        if not monthly:
            print(f"mlaapde: no monthly metadata files found in {MLAAPDE_DIR}")
            continue
        dfs = []
        for f in monthly:
            df = pd.read_csv(f, low_memory=False)
            df["source_month"] = f.stem.split("_")[1]   # e.g. '201307'
            dfs.append(df)
        df_all = pd.concat(dfs, ignore_index=True)
        dataset_meta["mlaapde"] = df_all.set_index("trace_name")
        print(f"mlaapde: {len(dataset_meta['mlaapde']):,} rows "
              f"from {len(monthly)} monthly files (Jul 2013 – Oct 2014)")
        continue

    # ── All other datasets: single metadata.csv via cache or download ─────
    local_path = METADATA_DIR / f"{ds_name}_metadata.csv"
    seisbench_cache = SEISBENCH_CACHE / ds_name / "metadata.csv"

    if seisbench_cache.exists():
        shutil.copy(seisbench_cache, local_path)
        print(f"{ds_name}: copied from local SeisBench cache")
    elif local_path.exists():
        print(f"{ds_name}: metadata already downloaded → reusing")
    else:
        url = SEISBENCH_META_URL.format(name=ds_name)
        print(f"{ds_name}: downloading from {url} ...", end=" ", flush=True)
        try:
            r = requests.get(url, stream=True, timeout=60)
            r.raise_for_status()
            with open(local_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
            print(f"done ({local_path.stat().st_size / 1e6:.1f} MB)")
        except Exception as e:
            print(f"FAILED: {e}")
            continue

    try:
        df = pd.read_csv(local_path, low_memory=False)
        dataset_meta[ds_name] = df.set_index("trace_name") if "trace_name" in df.columns else df
        print(f"  {ds_name}: {len(dataset_meta[ds_name]):,} rows")
    except Exception as e:
        print(f"  {ds_name}: could not parse — {e}")

print(f"\nMetadata loaded for {len(dataset_meta)} / {manifest['dataset'].nunique()} datasets")


instancecounts: copied from local SeisBench cache
  instancecounts: 1,159,249 rows
stead: copied from local SeisBench cache
  stead: 1,265,657 rows
pnw: copied from local SeisBench cache
  pnw: 183,909 rows
scedc: copied from local SeisBench cache
  scedc: 8,035,833 rows
txed: copied from local SeisBench cache
  txed: 519,689 rows
mlaapde: 228,256 rows from 16 monthly files (Jul 2013 – Oct 2014)
iquique: copied from local SeisBench cache
  iquique: 13,400 rows
obst2024: copied from local SeisBench cache
  obst2024: 60,394 rows

Metadata loaded for 8 / 8 datasets


## 2.3  Build FDSN Request List

Reconstructs (Network, Station, Location, Channel, starttime, endtime) for every
**non-MLAAPDE** trace in the manifest using the metadata CSV columns.

**MLAAPDE traces are skipped here** — their waveforms are already on disk and are
loaded directly from SeisBench in Section 2.6b. They appear in the request list
with `status = "mlaapde_local"` so the summary counts are still accurate.

Time resolution uses a three-level fallback:
1. `trace_start_time` (most datasets)
2. `source_origin_time` − buffer (TXED has only origin time)
3. Time parsed from `trace_name` or `trace_name_original` string


In [10]:
# ── Diagnostic: what time columns does each dataset actually have? ─────────
# Run this BEFORE cell 2.3 to understand why traces are missing start times.

TIME_CANDIDATES = [
    "trace_start_time", "starttime", "trace_start",
    "source_origin_time", "event_time", "origin_time",
    "trace_dt", "start_time", "time",
]

print(f"{'Dataset':20s}  {'Time column found':30s}  {'Sample value'}")
print("-" * 80)

for ds_name, df in dataset_meta.items():
    found = [(c, str(df[c].iloc[0])[:30]) for c in TIME_CANDIDATES if c in df.columns]
    if found:
        for col, sample in found:
            print(f"{ds_name:20s}  {col:30s}  {sample}")
    else:
        time_like = [c for c in df.columns if any(
            kw in c.lower() for kw in ["time", "date", "start", "origin"])]
        print(f"{ds_name:20s}  *** NONE FOUND ***  "
              f"Time-like cols: {time_like[:4]}")


Dataset               Time column found               Sample value
--------------------------------------------------------------------------------
instancecounts        trace_start_time                2016-11-16T01:35:48.20Z
instancecounts        source_origin_time              2016-11-16T01:35:56.21Z
stead                 trace_start_time                2015-10-21 05:55:00
stead                 source_origin_time              nan
pnw                   trace_start_time                2002-10-03T01:55:59.530000Z
pnw                   source_origin_time              2002-10-03T01:56:49.530000Z
scedc                 trace_start_time                2000-01-09T17:06:00.679900Z
scedc                 source_origin_time              2000-01-09T17:06:16.560000Z
txed                  source_origin_time              2022-07-03T20:06:21.490000Z
mlaapde               trace_start_time                2013-07-23T01:12:05.348400Z
mlaapde               source_origin_time              2013-07-23T01:12:5

In [11]:
# ── Column name aliases — expanded to cover all known SeisBench variants ──
NET_COLS  = ["station_network_code", "network",  "net",  "trace_network"]
STA_COLS  = ["station_code",         "station",  "sta",  "trace_station"]
LOC_COLS  = ["station_location_code","location", "loc",  "trace_location", "trace_loc"]
CHA_COLS  = ["trace_channel",        "station_channels", "channel", "cha",
             "trace_component",      "components"]
TIME_COLS = [
    "trace_start_time",       # most SeisBench datasets
    "starttime",
    "trace_start",
    "start_time",
    "trace_dt",
]
# Fallback: use event/origin time + offset when no trace start time exists.
# We request FETCH_DURATION seconds starting before origin so P is in the window.
ORIGIN_COLS = ["source_origin_time", "event_time", "origin_time", "eq_time"]
# For teleseismic events P can arrive 100-1000s after origin.
# We use p_arrival_sample / native_sr to compute the offset back from P.
ORIGIN_PRE_P_BUFFER = 30    # seconds before estimated P to start request

def first_col(df, candidates, default=""):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return pd.Series(default, index=df.index)

def parse_channels(ch_str):
    if pd.isna(ch_str): return "HH?"
    parts = str(ch_str).replace(";", ",").split(",")
    return parts[0][:2] + "?"

def extract_time_from_string(s):
    """
    Try every known SeisBench time format embedded in a string.
    Covers trace_name and trace_name_original for all datasets including MLAAPDE.
    Returns UTCDateTime or None.
    """
    import re
    if not s or str(s) in ("nan", "None", ""):
        return None
    s = str(s)

    # Format 1 — ISO with dashes: 2019-01-01T00:00:00
    m = re.search(r'(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})', s)
    if m:
        try: return UTCDateTime(m.group(1))
        except: pass

    # Format 2 — compact ISO (MLAAPDE / IRIS FDSN):
    #   __20190101T000000Z__  or  20190101T000000.000000Z
    m = re.search(r'(\d{8})T(\d{6})', s)
    if m:
        try:
            date, time = m.groups()
            return UTCDateTime(
                f"{date[:4]}-{date[4:6]}-{date[6:8]}"
                f"T{time[:2]}:{time[2:4]}:{time[4:6]}")
        except: pass

    # Format 3 — YYYY.DOY.HHMMSS: 2004.001.000000
    m = re.search(r'(\d{4})\.(\d{3})\.(\d{6})', s)
    if m:
        try:
            y, d, t = m.groups()
            return UTCDateTime(f"{y}-01-01") + (int(d)-1)*86400 + \
                   int(t[:2])*3600 + int(t[2:4])*60 + int(t[4:6])
        except: pass

    # Format 4 — plain date only: 2019-01-01  (treat as midnight)
    m = re.search(r'(\d{4}-\d{2}-\d{2})', s)
    if m:
        try: return UTCDateTime(m.group(1))
        except: pass

    return None

def extract_time_from_trace_name(trace_name):
    """Kept for back-compat — delegates to extract_time_from_string."""
    return extract_time_from_string(trace_name)

fdsn_requests = []

for _, row in manifest.iterrows():
    ds_name  = row["dataset"]
    tname    = row["trace_name"]
    p_sample = float(row["p_arrival_sample"])

    # MLAAPDE loaded locally from SeisBench HDF5 — never use FDSN
    if ds_name == "mlaapde":
        fdsn_requests.append({"trace_name": tname, "dataset": ds_name,
                               "status": "mlaapde_local"})
        continue

    if ds_name not in dataset_meta:
        fdsn_requests.append({"trace_name": tname, "dataset": ds_name,
                               "status": "no_metadata"})
        continue

    meta = dataset_meta[ds_name]
    if tname not in meta.index:
        fdsn_requests.append({"trace_name": tname, "dataset": ds_name,
                               "status": "not_in_metadata"})
        continue

    # NSLC
    net = str(first_col(meta, NET_COLS).get(tname, ""))
    sta = str(first_col(meta, STA_COLS).get(tname, ""))
    loc = str(first_col(meta, LOC_COLS).get(tname, ""))
    cha = parse_channels(first_col(meta, CHA_COLS).get(tname, "HH?"))
    loc = "" if loc in ("nan", "None", "--") else loc

    # Native SR (needed to compute offsets)
    sr_series  = meta.get("trace_sampling_rate_hz",
                           pd.Series(100.0, index=meta.index))
    native_sr  = float(sr_series.get(tname, 100.0))
    if pd.isna(native_sr) or native_sr <= 0:
        native_sr = 100.0

    # ── Resolve start time (three-level fallback) ──────────────────────────
    t_start = None

    # Level 1: direct trace start time column
    t_raw = first_col(meta, TIME_COLS).get(tname, None)
    if t_raw and str(t_raw) not in ("", "nan", "None"):
        try:
            t_start = UTCDateTime(str(t_raw))
        except Exception:
            pass

    # Level 2: origin time − travel time estimate
    # We know P arrives at p_sample samples into the trace.
    # p_sample / native_sr = seconds from trace_start to P.
    # So trace_start ≈ P_UTC_time - (p_sample / native_sr)
    # We estimate P_UTC from origin_time + a fixed lead (conservative).
    if t_start is None:
        orig_raw = first_col(meta, ORIGIN_COLS).get(tname, None)
        if orig_raw and str(orig_raw) not in ("", "nan", "None"):
            try:
                t_origin = UTCDateTime(str(orig_raw))
                # Offset so that P falls roughly at p_sample from the start.
                # We subtract ORIGIN_PRE_P_BUFFER to start a bit before P.
                p_seconds = p_sample / native_sr
                t_start   = t_origin - p_seconds + \
                            (p_sample / native_sr) - ORIGIN_PRE_P_BUFFER
                # Simplified: just start ORIGIN_PRE_P_BUFFER before P.
                # Actual: start = p_time - p_sample/sr, p_time ≈ origin + travel
                # Since we don't know travel time, start at origin - buffer.
                t_start = t_origin - ORIGIN_PRE_P_BUFFER
            except Exception:
                pass

    # Level 3: parse time from trace_name string itself
    if t_start is None:
        t_start = extract_time_from_string(tname)

    # Level 3b: some datasets embed timing in 'trace_name_original'.
    # e.g. "IU.ANMO.00.BHZ__20190101T000000Z__20190101T003000Z"
    # This catches any dataset where the original identifier carries the time.
    if t_start is None and "trace_name_original" in meta.columns:
        orig_name = meta["trace_name_original"].get(tname, None)
        t_start   = extract_time_from_string(orig_name)

    if t_start is None:
        fdsn_requests.append({"trace_name": tname, "dataset": ds_name,
                               "status": "no_start_time"})
        continue

    fdsn_requests.append({
        "trace_name":  tname,
        "dataset":     ds_name,
        "network":     net,
        "station":     sta,
        "location":    loc,
        "channel":     cha,
        "t_start":     t_start,
        "t_end":       t_start + FETCH_DURATION,
        "p_sample":    p_sample,
        "native_sr":   native_sr,
        "has_p_pick":  bool(row.get("has_p_pick", pd.notna(row["p_arrival_sample"]))),
        "has_s_pick":  bool(row["has_s_pick"]),
        "s_sample":    float(row["s_arrival_sample"]) if row["has_s_pick"] else np.nan,
        "status":      "pending",
        "fdsn_client": DS_FDSN_CLIENT.get(ds_name, "IRIS"),
    })

req_df  = pd.DataFrame(fdsn_requests)
pending = req_df[req_df["status"] == "pending"]
problem = req_df[req_df["status"] != "pending"]

mlaapde_local = req_df[req_df["status"] == "mlaapde_local"]
print(f"Requests built : {len(pending):,} FDSN pending")
print(f"MLAAPDE local  : {len(mlaapde_local):,} (loaded from SeisBench HDF5 in section 2.6b)")
print(f"Skipped        : {len(req_df[~req_df['status'].isin(['pending','mlaapde_local'])]):,}")
if len(problem := req_df[~req_df['status'].isin(['pending','mlaapde_local'])]):
    print("\nSkip reasons:"); print(problem["status"].value_counts().to_string())
print()
print(pending.groupby(["dataset","fdsn_client"])["trace_name"]
      .count().rename("n_requests").to_string())


Requests built : 3,126 FDSN pending
MLAAPDE local  : 1,350 (loaded from SeisBench HDF5 in section 2.6b)
Skipped        : 0

dataset         fdsn_client
instancecounts  INGV           845
iquique         IRIS           320
obst2024        IRIS           400
pnw             IRIS           378
scedc           SCEDC          326
stead           NCEDC          672
txed            IRIS           185


## 2.4  Execute Bulk FDSN Requests

Fetches non-MLAAPDE waveforms in batches of `FDSN_BATCH` traces per request.
Each trace is saved as a MiniSEED file in `waveform_cache/` so this cell
is safe to re-run — already-downloaded traces are skipped.

**MLAAPDE traces are not fetched here.** They are loaded directly from the
local `waveforms_YYYYMM.hdf5` files in Section 2.6b.

**Expected time:** ~0.5–2 s per batch of 50 traces.
For ~3,100 non-MLAAPDE traces (~62 batches) this takes roughly 5–15 minutes.


In [6]:
# ── Open one FDSN client per data centre ──────────────────────────────────
clients = {}
for dc in pending["fdsn_client"].unique():
    try:
        clients[dc] = Client(dc)
        print(f"Connected: {dc}")
    except Exception as e:
        print(f"WARNING: could not connect to {dc}: {e}")

# ── Helper: save a stream component to MiniSEED ───────────────────────────
def mseed_path(trace_name):
    """Sanitise trace_name for use as a filename."""
    safe = trace_name.replace("/", "_").replace("\\", "_").replace(" ", "_")
    return WAVEFORM_CACHE / f"{safe}.mseed"

# ── Bulk FDSN fetch ────────────────────────────────────────────────────────
results = {}   # trace_name → "ok" | "no_data" | "error:<msg>"

for dc, grp in pending.groupby("fdsn_client"):
    if dc not in clients:
        for tname in grp["trace_name"]:
            results[tname] = f"error:client_{dc}_unavailable"
        continue

    client = clients[dc]
    rows   = grp.to_dict("records")

    # Split into batches
    batches = [rows[i:i+FDSN_BATCH] for i in range(0, len(rows), FDSN_BATCH)]

    for batch in tqdm(batches, desc=f"FDSN {dc}", unit="batch"):
        # Separate already-cached traces
        to_fetch = [r for r in batch if not mseed_path(r["trace_name"]).exists()]
        for r in batch:
            if mseed_path(r["trace_name"]).exists():
                results[r["trace_name"]] = "ok_cached"

        if not to_fetch:
            continue

        # Build bulk request tuples
        bulk = [
            (r["network"], r["station"], r["location"],
             r["channel"], r["t_start"], r["t_end"])
            for r in to_fetch
        ]

        try:
            st = client.get_waveforms_bulk(bulk, attach_response=False)
        except FDSNNoDataException:
            for r in to_fetch:
                results[r["trace_name"]] = "no_data"
            continue
        except Exception as e:
            for r in to_fetch:
                results[r["trace_name"]] = f"error:{str(e)[:60]}"
            continue

        # Match returned traces back to request rows and save
        for r in to_fetch:
            tname = r["trace_name"]
            # Select components for this station
            sub = st.select(network=r["network"], station=r["station"],
                            channel=r["channel"].replace("?", "*"))
            if len(sub) == 0:
                results[tname] = "no_data"
            else:
                try:
                    sub.write(str(mseed_path(tname)), format="MSEED")
                    results[tname] = "ok"
                except Exception as e:
                    results[tname] = f"save_error:{str(e)[:40]}"

# Summary
ok_count  = sum(1 for v in results.values() if v.startswith("ok"))
err_count = len(results) - ok_count
print(f"\nFDSN fetch complete: {ok_count:,} ok  |  {err_count:,} failed/no-data")
if err_count:
    from collections import Counter
    err_types = Counter(v for v in results.values() if not v.startswith("ok"))
    for reason, count in err_types.most_common(10):
        print(f"  {count:>5,} — {reason}")


Connected: INGV
Connected: NCEDC
Connected: IRIS
Connected: SCEDC


FDSN INGV:   0%|          | 0/17 [00:00<?, ?batch/s]

FDSN IRIS:   0%|          | 0/26 [00:00<?, ?batch/s]

FDSN NCEDC:   0%|          | 0/14 [00:00<?, ?batch/s]

FDSN SCEDC:   0%|          | 0/7 [00:00<?, ?batch/s]


FDSN fetch complete: 1,086 ok  |  2,040 failed/no-data
  2,040 — no_data


## 2.5  Preprocessing Functions

Core pipeline used by both Section 2.6 (MiniSEED) and 2.6b (MLAAPDE):

| Function | Purpose |
|----------|---------|
| `resample_waveform` | Polyphase resampling to TARGET_SR — handles 40 Hz MLAAPDE and any other non-100 Hz source |
| `extract_window` | 30 s window with random P-jitter; can anchor on P **or S** (needed for MLAAPDE S-only trace) |
| `normalize_waveform` | Zero-mean, unit-std per component; silent channels left as zeros |
| `compute_snr_db` | Pre-P noise vs post-P signal window |
| `check_quality` | Detects NaN, flat-line channels, too many zeros |


In [ ]:
from math import gcd

def resample_waveform(data, orig_sr, target_sr=TARGET_SR):
    """Polyphase resample (n_comp, n_samples) from orig_sr → target_sr."""
    if abs(orig_sr - target_sr) < 0.5:
        return data.astype(np.float32)
    up   = int(round(target_sr))
    down = int(round(orig_sr))
    g    = gcd(up, down);  up //= g;  down //= g
    return scipy.signal.resample_poly(data, up, down, axis=-1,
                                      padtype="line").astype(np.float32)

def rescale_sample(sample, orig_sr, target_sr=TARGET_SR):
    return int(round(sample * target_sr / orig_sr))

def extract_window(waveform, anchor_samp, other_samp=None,
                   wlen=WINDOW_SAMPLES, pmin=P_JITTER_MIN, pmax=P_JITTER_MAX, rng=rng):
    """
    Extract a waveform window centred on `anchor_samp` (normally P, but S for S-only traces).
    `other_samp` is the position of the secondary phase (S if anchoring on P, or vice versa).
    Returns (window, anchor_in_window, other_in_window) where other_in_window = -1 if absent.
    """
    n = waveform.shape[-1]
    offset    = int(rng.integers(pmin, pmax))   # anchor placed randomly in [pmin, pmax]
    win_start = anchor_samp - offset
    win_end   = win_start + wlen
    if win_start < 0 or win_end > n:
        pad_l = max(0, -win_start);  pad_r = max(0, win_end - n)
        waveform  = np.pad(waveform, ((0,0),(pad_l,pad_r)))
        win_start += pad_l;  win_end += pad_l
    window = waveform[:, win_start:win_end].astype(np.float32)
    other_in_win = -1
    if other_samp is not None and np.isfinite(other_samp):
        other_in_win = int(round(other_samp)) - win_start
        if not (0 <= other_in_win < wlen):
            other_in_win = -1
    return window, offset, other_in_win

def normalize_waveform(w, eps=1e-10):
    out = np.zeros_like(w, dtype=np.float32)
    for i in range(w.shape[0]):
        c = w[i].astype(np.float64)
        std = np.nanstd(c)
        if std > eps:
            out[i] = ((c - np.nanmean(c)) / std).astype(np.float32)
    return out

def compute_snr_db(w, p_pos, noise_len=200, signal_len=200):
    n_comp, n_samp = w.shape
    ns = max(0, p_pos - noise_len);  ne = p_pos
    ss = p_pos;  se = min(n_samp, p_pos + signal_len)
    if ne <= ns or se <= ss:
        return np.nan
    snrs = []
    for c in range(n_comp):
        nr = np.sqrt(np.mean(w[c,ns:ne]**2))
        sr = np.sqrt(np.mean(w[c,ss:se]**2))
        if nr > 1e-12:
            snrs.append(20*np.log10(sr/nr + 1e-12))
    return float(np.mean(snrs)) if snrs else np.nan

def check_quality(w, min_nonzero=0.8):
    if np.any(np.isnan(w)):
        return False, "contains_nan"
    if np.count_nonzero(w)/w.size < min_nonzero:
        return False, "too_many_zeros"
    for i in range(w.shape[0]):
        if np.std(w[i]) < 1e-12:
            return False, f"flat_component_{i}"
    return True, None

print("Preprocessing functions ready.")


## 2.6  Main Loop: MiniSEED → HDF5 (non-MLAAPDE)

Processes all **non-MLAAPDE** traces: reads each MiniSEED file from
`waveform_cache/`, resamples if needed, extracts a 30 s window, normalises,
computes SNR, and writes to `benchmark_waveforms.hdf5`.

MLAAPDE traces are processed in Section 2.6b immediately after this cell.
The HDF5 file is opened in write mode here and in append mode in 2.6b.


In [ ]:
skip_log    = []
success_log = []

# Build a lookup from pending requests
req_lookup = {r["trace_name"]: r for r in pending.to_dict("records")}

with h5py.File(OUTPUT_HDF5, "w") as hf:
    wf_grp  = hf.create_group("waveforms")
    p_grp   = hf.create_group("p_sample")
    s_grp   = hf.create_group("s_sample")
    snr_grp = hf.create_group("snr_db")

    # Filter manifest to non-MLAAPDE traces only
    non_mlaapde = manifest[manifest["dataset"] != "mlaapde"]

    for _, row in tqdm(non_mlaapde.iterrows(), total=len(non_mlaapde), desc="Non-MLAAPDE"):
        tname   = row["trace_name"]
        ds_name = row["dataset"]

        # ── 1. Check FDSN fetch outcome ───────────────────────────────────
        fetch_status = results.get(tname, req_lookup.get(tname, {}).get("status", "not_requested"))
        if not fetch_status.startswith("ok") and not mseed_path(tname).exists():
            skip_log.append({"trace_name": tname, "dataset": ds_name,
                              "reason": f"fdsn:{fetch_status}"})
            continue

        # ── 2. Load MiniSEED ──────────────────────────────────────────────
        mpath = mseed_path(tname)
        if not mpath.exists():
            skip_log.append({"trace_name": tname, "dataset": ds_name,
                              "reason": "mseed_file_missing"})
            continue
        try:
            from obspy import read as obspy_read
            st = obspy_read(str(mpath))
        except Exception as e:
            skip_log.append({"trace_name": tname, "dataset": ds_name,
                              "reason": f"mseed_read_error:{str(e)[:40]}"})
            continue

        # ── 3. Convert Stream to ndarray (Z, N, E order where possible) ───
        # Merge any gaps, sort by channel
        st.merge(fill_value=0)
        st.sort(["channel"])
        traces_data = [tr.data.astype(np.float32) for tr in st]
        if not traces_data:
            skip_log.append({"trace_name": tname, "dataset": ds_name,
                              "reason": "empty_stream"})
            continue

        # Pad/stack to (3, max_len); single-component → 3 identical channels
        max_len  = max(len(d) for d in traces_data)
        waveform = np.zeros((3, max_len), dtype=np.float32)
        for i, d in enumerate(traces_data[:3]):
            waveform[i, :len(d)] = d

        # ── 4. Resample ───────────────────────────────────────────────────
        orig_sr = req_lookup.get(tname, {}).get("native_sr", 100.0)
        actual_sr = float(st[0].stats.sampling_rate) if st else orig_sr
        if abs(actual_sr - TARGET_SR) > 0.5:
            try:
                waveform = resample_waveform(waveform, actual_sr, TARGET_SR)
            except Exception as e:
                skip_log.append({"trace_name": tname, "dataset": ds_name,
                                  "reason": f"resample_error:{str(e)[:40]}"})
                continue

        # ── 5. Rescale P/S samples to TARGET_SR coords ────────────────────
        # Non-MLAAPDE traces always have P (guaranteed by load_and_filter in Step 1)
        p_raw  = float(row["p_arrival_sample"])
        s_raw  = float(row["s_arrival_sample"]) if row["has_s_pick"] else np.nan
        p_samp = rescale_sample(p_raw, orig_sr, TARGET_SR)
        s_samp = rescale_sample(s_raw, orig_sr, TARGET_SR) if np.isfinite(s_raw) else None

        n_resampled = waveform.shape[-1]
        if not (P_JITTER_MIN <= p_samp <= n_resampled - (WINDOW_SAMPLES - P_JITTER_MAX)):
            skip_log.append({"trace_name": tname, "dataset": ds_name,
                              "reason": f"p_near_edge(p={p_samp},len={n_resampled})"})
            continue

        # ── 6. Window anchored on P, SNR computed on P window ─────────────
        window, p_in_win, s_in_win = extract_window(waveform, p_samp, s_samp)
        ok, reason = check_quality(window)
        if not ok:
            skip_log.append({"trace_name": tname, "dataset": ds_name,
                              "reason": f"quality:{reason}"})
            continue
        snr = compute_snr_db(window, p_in_win)
        window = normalize_waveform(window)

        # ── 7. Write to HDF5 ──────────────────────────────────────────────
        wf_grp.create_dataset(tname, data=window, compression="gzip",
                               compression_opts=4, dtype="float32")
        p_grp.create_dataset(tname,  data=np.int16(p_in_win))
        s_grp.create_dataset(tname,  data=np.int16(s_in_win))
        snr_grp.create_dataset(tname, data=np.float32(snr))
        success_log.append({"trace_name": tname, "dataset": ds_name,
                             "p_in_window": p_in_win, "s_in_window": s_in_win,
                             "snr_db": snr, "actual_sr": actual_sr})

print(f"\nDone.  Written: {len(success_log):,}  |  Skipped: {len(skip_log):,}")


## 2.6b  MLAAPDE Local HDF5 → HDF5

Processes all 1,350 MLAAPDE teleseismic traces directly from the local
SeisBench HDF5 files — no FDSN download needed.

**Key differences from Section 2.6:**
- Waveforms loaded via `sbd.MLAAPDE(sampling_rate=None)` at native 40 Hz
- Resampled to 100 Hz (P/S sample positions rescaled accordingly)
- **P-only traces (1,349):** window anchored on P; `s_in_window = -1`
- **S-only traces (1):** window anchored on S; `p_in_window = -1`
- HDF5 opened in **append mode** so non-MLAAPDE traces written in 2.6 are preserved


In [ ]:
# ── 2.6b  MLAAPDE: local SeisBench HDF5 → preprocess → HDF5 ──────────────
import seisbench.data as sbd

mlaapde_manifest = manifest[manifest["dataset"] == "mlaapde"]
print(f"MLAAPDE traces to process: {len(mlaapde_manifest):,}")
if len(mlaapde_manifest) == 0:
    print("No MLAAPDE traces in manifest — skipping.")
else:
    # ── Load SeisBench MLAAPDE index once (handles sharding internally) ────
    print("Building MLAAPDE SeisBench index (metadata only)...", end=" ", flush=True)
    ds_mlaapde      = sbd.MLAAPDE(sampling_rate=None)   # native 40 Hz
    mlaapde_name_to_pos = {name: pos for pos, name in
                            enumerate(ds_mlaapde.metadata["trace_name"])}
    print(f"{len(mlaapde_name_to_pos):,} traces indexed")

    # ── Append MLAAPDE traces into the existing HDF5 ──────────────────────
    with h5py.File(OUTPUT_HDF5, "a") as hf:   # append — preserves non-MLAAPDE data
        wf_grp  = hf.require_group("waveforms")
        p_grp   = hf.require_group("p_sample")
        s_grp   = hf.require_group("s_sample")
        snr_grp = hf.require_group("snr_db")

        n_ok = 0;  n_skip = 0

        for _, row in tqdm(mlaapde_manifest.iterrows(),
                           total=len(mlaapde_manifest), desc="MLAAPDE"):
            tname = row["trace_name"]

            if tname in wf_grp:           # already written — safe to re-run
                n_ok += 1;  continue

            # ── 1. Determine pick availability ────────────────────────────
            has_p = bool(row.get("has_p_pick", pd.notna(row["p_arrival_sample"])))
            has_s = bool(row["has_s_pick"])

            if not has_p and not has_s:
                skip_log.append({"trace_name": tname, "dataset": "mlaapde",
                                  "reason": "no_picks"})
                n_skip += 1;  continue

            # ── 2. Load waveform via SeisBench ────────────────────────────
            if tname not in mlaapde_name_to_pos:
                skip_log.append({"trace_name": tname, "dataset": "mlaapde",
                                  "reason": "not_in_seisbench_index"})
                n_skip += 1;  continue
            try:
                waveform, _ = ds_mlaapde[mlaapde_name_to_pos[tname]]
                waveform    = waveform.astype(np.float32)
                if waveform.ndim == 1:
                    waveform = waveform[np.newaxis, :]
            except Exception as e:
                skip_log.append({"trace_name": tname, "dataset": "mlaapde",
                                  "reason": f"load_error:{str(e)[:50]}"})
                n_skip += 1;  continue

            # ── 3. Resample 40 Hz → 100 Hz ───────────────────────────────
            try:
                waveform = resample_waveform(waveform, MLAAPDE_NATIVE_SR, TARGET_SR)
            except Exception as e:
                skip_log.append({"trace_name": tname, "dataset": "mlaapde",
                                  "reason": f"resample_error:{str(e)[:40]}"})
                n_skip += 1;  continue

            # ── 4. Rescale arrival samples from 40 Hz → 100 Hz coords ─────
            p_raw = float(row["p_arrival_sample"]) if has_p else np.nan
            s_raw = float(row["s_arrival_sample"]) if has_s else np.nan
            p_samp_100 = rescale_sample(p_raw, MLAAPDE_NATIVE_SR, TARGET_SR) if has_p else None
            s_samp_100 = rescale_sample(s_raw, MLAAPDE_NATIVE_SR, TARGET_SR) if has_s else None

            n_resampled = waveform.shape[-1]

            # ── 5. Extract window — anchor on P (or S for S-only traces) ──
            if has_p:
                # Normal case: anchor on P, record where S lands (if any)
                if not (P_JITTER_MIN <= p_samp_100 <= n_resampled - (WINDOW_SAMPLES - P_JITTER_MAX)):
                    skip_log.append({"trace_name": tname, "dataset": "mlaapde",
                                      "reason": f"p_near_edge(p={p_samp_100},len={n_resampled})"})
                    n_skip += 1;  continue
                window, p_in_win, s_in_win = extract_window(
                    waveform, p_samp_100, s_samp_100)
            else:
                # S-only: anchor on S, p_in_window = -1
                if not (P_JITTER_MIN <= s_samp_100 <= n_resampled - (WINDOW_SAMPLES - P_JITTER_MAX)):
                    skip_log.append({"trace_name": tname, "dataset": "mlaapde",
                                      "reason": f"s_near_edge(s={s_samp_100},len={n_resampled})"})
                    n_skip += 1;  continue
                window, s_in_win, _ = extract_window(waveform, s_samp_100, None)
                p_in_win = -1

            # ── 6. Quality → SNR → normalise → write ─────────────────────
            ok, reason = check_quality(window)
            if not ok:
                skip_log.append({"trace_name": tname, "dataset": "mlaapde",
                                  "reason": f"quality:{reason}"})
                n_skip += 1;  continue

            # SNR computed on the P window if P available, else S window
            snr_ref  = p_in_win if p_in_win >= 0 else s_in_win
            snr      = compute_snr_db(window, snr_ref)
            window   = normalize_waveform(window)

            wf_grp.create_dataset(tname,  data=window, compression="gzip",
                                   compression_opts=4, dtype="float32")
            p_grp.create_dataset(tname,   data=np.int16(p_in_win))
            s_grp.create_dataset(tname,   data=np.int16(s_in_win))
            snr_grp.create_dataset(tname, data=np.float32(snr))
            success_log.append({"trace_name": tname, "dataset": "mlaapde",
                                  "p_in_window": p_in_win, "s_in_window": s_in_win,
                                  "snr_db": snr, "actual_sr": TARGET_SR})
            n_ok += 1

    print(f"MLAAPDE: {n_ok:,} written | {n_skip:,} skipped")
    print(f"Total written so far: {len(success_log):,}")


## 2.7  Save Processing Index

In [ ]:
success_df = pd.DataFrame(success_log)
skip_df    = pd.DataFrame(skip_log) if skip_log else pd.DataFrame(
                 columns=["trace_name","dataset","reason"])
success_df["status"] = "ok";  skip_df["status"] = "skipped"

index_df = manifest.merge(
    success_df[["trace_name","p_in_window","s_in_window","snr_db","actual_sr","status"]],
    on="trace_name", how="left")
index_df["status"] = index_df["status"].fillna("skipped")
skip_reason = skip_df.set_index("trace_name")["reason"] if len(skip_df) else pd.Series(dtype=str)
index_df["skip_reason"] = index_df["trace_name"].map(skip_reason)
index_df.to_csv("benchmark_waveforms_index.csv", index=False)
print(f"Saved benchmark_waveforms_index.csv ({len(index_df):,} rows)")
if len(skip_df):
    print("\nTop skip reasons:")
    print(skip_df["reason"].str.split(":").str[0].value_counts().head(10).to_string())


## 2.8  Quality Summary

In [ ]:
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec

idx = pd.read_csv("benchmark_waveforms_index.csv")
ok  = idx[idx["status"] == "ok"]
print(f"Written: {len(ok):,} / {len(idx):,}  ({100*len(ok)/len(idx):.1f}%)")
print(ok.groupby("dataset")["trace_name"].count().rename("n_written").to_string())

fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle("Step 2 — Quality Summary", fontsize=14, fontweight="bold")
COLORS = plt.cm.tab10.colors

# Written per dataset
ax1 = fig.add_subplot(gs[0,0])
ok.groupby("dataset")["trace_name"].count().sort_values().plot(
    kind="barh", ax=ax1, color="steelblue")
ax1.set_title("Traces Written per Dataset"); ax1.set_xlabel("Count")

# SNR distribution
ax2 = fig.add_subplot(gs[0,1])
for i,(ds,grp) in enumerate(ok.groupby("dataset")):
    ax2.hist(grp["snr_db"].dropna(), bins=30, alpha=0.5, density=True,
             label=ds, color=COLORS[i%10])
ax2.set_xlabel("SNR (dB)"); ax2.set_title("SNR Distribution"); ax2.legend(fontsize=7,ncol=2)

# P position in window
ax3 = fig.add_subplot(gs[0,2])
ax3.hist(ok["p_in_window"].dropna(), bins=50, color="steelblue", edgecolor="white")
ax3.axvline(P_JITTER_MIN,color="red",linestyle="--",lw=1,label=f"min={P_JITTER_MIN}")
ax3.axvline(P_JITTER_MAX,color="orange",linestyle="--",lw=1,label=f"max={P_JITTER_MAX}")
ax3.set_xlabel("P sample in window"); ax3.set_title("P Position in Window"); ax3.legend(fontsize=8)

# S position
ax4 = fig.add_subplot(gs[1,0])
has_s = ok[ok["s_in_window"]>=0]
ax4.hist(has_s["s_in_window"],bins=50,color="darkorange",edgecolor="white")
ax4.set_xlabel("S sample in window"); ax4.set_title(f"S Position (n={len(has_s):,} P+S traces)")

# Written vs skipped
ax5 = fig.add_subplot(gs[1,1])
ds_tot = idx.groupby("dataset")["trace_name"].count()
ds_ok  = ok.groupby("dataset")["trace_name"].count().reindex(ds_tot.index,fill_value=0)
x = np.arange(len(ds_tot))
ax5.bar(x, ds_ok, label="Written", color="steelblue")
ax5.bar(x, ds_tot-ds_ok, bottom=ds_ok, label="Skipped", color="tomato",alpha=0.7)
ax5.set_xticks(x); ax5.set_xticklabels(ds_tot.index,rotation=30,ha="right",fontsize=8)
ax5.set_title("Written vs Skipped"); ax5.legend(fontsize=8)

# SNR vs distance bin
ax6 = fig.add_subplot(gs[1,2])
bins = ok["dist_bin"].unique()
for i,(b,grp) in enumerate(ok.groupby("dist_bin")):
    ax6.violinplot([grp["snr_db"].dropna().values], positions=[i],
                   showmedians=True, widths=0.6)
ax6.set_xticks(range(len(bins)))
ax6.set_xticklabels(bins,rotation=20,ha="right",fontsize=8)
ax6.set_ylabel("SNR (dB)"); ax6.set_title("SNR vs Distance Bin")

plt.savefig("step2_quality_summary.png",dpi=150,bbox_inches="tight")
plt.show(); print("Saved → step2_quality_summary.png")


## 2.9  Example Waveform Verification

In [ ]:
idx = pd.read_csv("benchmark_waveforms_index.csv")
ok  = idx[idx["status"]=="ok"]
n_ds = ok["dataset"].nunique()
fig, axes = plt.subplots(n_ds, 1, figsize=(14, 2.8*n_ds))
if n_ds == 1: axes = [axes]
fig.suptitle("Example Waveforms from HDF5 (Z component)", fontsize=13, fontweight="bold")

with h5py.File(OUTPUT_HDF5,"r") as hf:
    for ax, (ds_name, grp) in zip(axes, ok.groupby("dataset")):
        row = grp.iloc[(grp["snr_db"]-grp["snr_db"].median()).abs().argsort().iloc[0]]
        tname = row["trace_name"]
        if tname not in hf["waveforms"]:
            ax.set_title(f"{ds_name} — not found in HDF5"); continue
        wave  = hf["waveforms"][tname][:]
        p_pos = int(hf["p_sample"][tname][()])
        s_pos = int(hf["s_sample"][tname][()])
        snr   = float(hf["snr_db"][tname][()])
        t = np.arange(WINDOW_SAMPLES)/TARGET_SR
        ax.plot(t, wave[0], lw=0.6, color="steelblue")
        ax.axvline(p_pos/TARGET_SR, color="red", lw=1.5, label=f"P ({p_pos/TARGET_SR:.1f}s)")
        if s_pos >= 0:
            ax.axvline(s_pos/TARGET_SR, color="darkorange", lw=1.5, label=f"S ({s_pos/TARGET_SR:.1f}s)")
        dist_str = f"{row['distance_km']:.0f} km" if pd.notna(row.get("distance_km")) else "?"
        ax.set_title(f"{ds_name} | dist={dist_str} | SNR={snr:.1f} dB", fontsize=9)
        ax.set_xlim(0, WINDOW_SAMPLES/TARGET_SR); ax.set_ylabel("Norm. amp.")
        ax.legend(fontsize=7, loc="upper right")
axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.savefig("step2_example_waveforms.png",dpi=150,bbox_inches="tight")
plt.show(); print("Saved → step2_example_waveforms.png")


## 2.10  Outputs & Usage in Step 3

| File | Size | Description |
|------|------|-------------|
| `benchmark_waveforms.hdf5` | ~600 MB | Preprocessed waveforms, P/S positions, SNR — all 8 datasets including MLAAPDE |
| `benchmark_waveforms_index.csv` | ~1 MB | Per-trace status, skip reasons, SNR |
| `waveform_cache/` | ~2 GB | Raw MiniSEED per non-MLAAPDE trace |
| `seisbench_metadata/` | ~50 MB | Metadata CSVs (single files for most datasets; MLAAPDE loaded directly from cache) |

### Loading in Step 3

```python
import h5py, pandas as pd

index = pd.read_csv("benchmark_waveforms_index.csv")
ok    = index[index["status"] == "ok"]

# All traces (P or S evaluation)
all_traces = ok

# P-recall evaluation — all traces with a P pick
p_traces  = ok[ok["p_in_window"] >= 0]

# S-recall evaluation — only traces where S falls within the window
s_traces  = ok[ok["s_in_window"] >= 0]

# Ts-Tp meaningful only when both P and S are in the window
ps_traces = ok[(ok["p_in_window"] >= 0) & (ok["s_in_window"] >= 0)]

with h5py.File("benchmark_waveforms.hdf5", "r") as hf:
    wave    = hf["waveforms"]["my_trace_name"][:]   # (3, 3000) float32
    p_pos   = int(hf["p_sample"]["my_trace_name"][()])   # -1 if no P in window
    s_pos   = int(hf["s_sample"]["my_trace_name"][()])   # -1 if no S in window
```

### MLAAPDE-specific notes for Step 3

- `p_in_window = -1` for the 1 S-only trace; `s_in_window = -1` for the 1,349 P-only traces
- All MLAAPDE waveforms are at 100 Hz in the HDF5 (resampled from 40 Hz in this notebook)
- P-recall can be evaluated on all 4,475 traces with `p_in_window >= 0`
- S-recall can only be evaluated on the ~3,126 traces with `s_in_window >= 0`
